In [1]:
!pip install -U sentence-transformers xgboost scikit-learn pandas matplotlib seaborn imbalanced-learn


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. LOAD
df = pd.read_csv('/Users/tsovinarbabakhanyan/Desktop/Armenian Chat part/src/data/legal_analysis_labeled.csv').dropna(subset=['Verdict_Text', 'Decision_Label'])

# 2. EMBED + PCA (Reduce 768 -> 20 dimensions)
model = SentenceTransformer('all-mpnet-base-v2')
embeddings = model.encode(df['Verdict_Text'].tolist(), show_progress_bar=True)
pca = PCA(n_components=20, random_state=42)
X = pca.fit_transform(embeddings)

le = LabelEncoder()
y = le.fit_transform(df['Decision_Label'])

# 3. XGBOOST (Tuned for Tiny Data)
clf = XGBClassifier(
    n_estimators=50,
    max_depth=3,          # Keep it shallow so it doesn't memorize
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=1,   # Balanced approach
    objective='multi:softprob'
)

# 4. CROSS-VAL
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
all_preds = np.zeros(len(y))

for train_idx, val_idx in skf.split(X, y):
    clf.fit(X[train_idx], y[train_idx])
    all_preds[val_idx] = clf.predict(X[val_idx])

print(f"XGBoost Accuracy: {accuracy_score(y, all_preds)*100:.2f}%")
print(classification_report(y, all_preds, target_names=le.classes_))

/Users/tsovinarbabakhanyan/Desktop/Armenian Chat part/rag_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0511 18:01:31.008000 19725 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Batches: 100%|██████████| 65/65 [04:52<00:00,  4.50s/it]
/Users/tsovinarbabakhanyan/Desktop/Armenian Chat part/rag_env/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [18:06:32] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/tsovinarbabakhanyan/Desktop/Armenian Chat part/rag_env/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [18:06:33] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.

XGBoost Accuracy: 68.55%
                              precision    recall  f1-score   support

                    Approved       0.71      0.54      0.61       847
 Partially Approved / Closed       0.67      0.86      0.76      1109
                    Rejected       0.73      0.07      0.13       110
Unknown / Needs Manual Check       0.00      0.00      0.00         7

                    accuracy                           0.69      2073
                   macro avg       0.53      0.37      0.38      2073
                weighted avg       0.69      0.69      0.66      2073



/Users/tsovinarbabakhanyan/Desktop/Armenian Chat part/rag_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/tsovinarbabakhanyan/Desktop/Armenian Chat part/rag_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/tsovinarbabakhanyan/Desktop/Armenian Chat part/rag_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_divisio

In [3]:
import json

# Assuming report_df and report_dict are already created from the classification report
report_dict = classification_report(y, all_preds, target_names=le.classes_, output_dict=True)
report_df = pd.DataFrame(report_dict).transpose()
report_df.to_csv('/Users/tsovinarbabakhanyan/Desktop/Armenian Chat part/src/data/classification_report.csv', index=True)

with open('/Users/tsovinarbabakhanyan/Desktop/Armenian Chat part/src/data/classification_report.json', 'w', encoding='utf-8') as f:
    json.dump(report_dict, f, indent=2, ensure_ascii=False)

print("Saved report_df to classification_report.csv and report_dict to classification_report.json")

Saved report_df to classification_report.csv and report_dict to classification_report.json


/Users/tsovinarbabakhanyan/Desktop/Armenian Chat part/rag_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/tsovinarbabakhanyan/Desktop/Armenian Chat part/rag_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/tsovinarbabakhanyan/Desktop/Armenian Chat part/rag_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_divisio